In [2]:
from idlelib import history

import nltk
nltk.download('gutenberg')

/home/sidd/miniconda3/envs/dl/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/sidd/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package gutenberg to /home/sidd/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


True

In [4]:
from nltk.corpus import gutenberg
import pandas as pd


In [5]:
data=gutenberg.raw('shakespeare-hamlet.txt')

In [6]:
with open('hamlet.txt','w') as f:
    f.write(data)

In [8]:
import numpy
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

I0000 00:00:1790577172.173587   39389 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790577172.174407   39389 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790577172.305127   39389 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790577173.346257   39389 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [9]:
with open('hamlet.txt','r') as f:
    text=f.read().lower()

In [10]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index) + 1
total_words

4818

In [11]:
input_sequences=[]
for line in text.split('\n'):
    token_list=tokenizer.texts_to_sequences([line])[0]
    for i in range(1,len(token_list)):
        n_grams_sequence=token_list[:i+1]
        input_sequences.append(n_grams_sequence)

In [12]:
#PAd sequences

max_sequence_len=max([len(x) for x in input_sequences])
max_sequence_len

14

In [13]:
input_sequences=numpy.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='post'))
input_sequences

array([[   1,  687,    0, ...,    0,    0,    0],
       [   1,  687,    4, ...,    0,    0,    0],
       [   1,  687,    4, ...,    0,    0,    0],
       ...,
       [4817,    1,  687, ...,    0,    0,    0],
       [4817,    1,  687, ...,    0,    0,    0],
       [4817,    1,  687, ...,    0,    0,    0]],
      shape=(25732, 14), dtype=int32)

In [14]:
X,y= input_sequences[:,:-1],input_sequences[:,-1]

In [15]:
import tensorflow as tf

y=tf.keras.utils.to_categorical(y,num_classes=total_words)

In [16]:
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Embedding, Dropout, LSTM

model = Sequential([
    Input(shape=(max_sequence_len,)),
    Embedding(total_words, 100),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 14, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 14, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history=model.fit(X_train,y_train,epochs=100,validation_data=(X_test,y_test),verbose=1)

Epoch 1/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.9983 - loss: 0.2925 - val_accuracy: 0.9998 - val_loss: 0.0046
Epoch 2/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.9998 - loss: 0.0035 - val_accuracy: 0.9998 - val_loss: 0.0037
Epoch 3/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.9998 - loss: 0.0028 - val_accuracy: 0.9998 - val_loss: 0.0036
Epoch 4/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9998 - loss: 0.0025 - val_accuracy: 0.9998 - val_loss: 0.0035
Epoch 5/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9998 - loss: 0.0025 - val_accuracy: 0.9998 - val_loss: 0.0036
Epoch 6/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9998 - loss: 0.0024 - val_accuracy: 0.9998 - val_loss: 0.0036
Epoch 7/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.9998 - loss: 0.0024 - val_accuracy: 0.9998 - val_loss: 0.0037
Epoch 8/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.9998 - loss: 0

In [30]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_word(model, tokenizer, input_text):
    max_len = model.input_shape[1]

    token_list = tokenizer.texts_to_sequences([input_text])[0]



    token_list = pad_sequences(
        [token_list],
        maxlen=max_len,
        padding='pre'
    )

    predicted = model.predict(token_list, verbose=0)

    # Ignore index 0 (padding)
    predicted_word_index = np.argmax(predicted[0][1:]) + 1

    predicted_word = tokenizer.index_word.get(
        predicted_word_index,
        "Unknown"
    )

    return predicted_word

In [31]:
tests = [
    "To be or not to be",
    "I may or may not be",
    "the quick brown",
    "once upon a",
    "it is a"
]

for text in tests:
    print(text, "→", predict_next_word(model, tokenizer, text))

To be or not to be → it
I may or may not be → could
the quick brown → it
once upon a → it
it is a → could


In [32]:
model.save('next_word_lstm.keras')

import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f,protocol=pickle.HIGHEST_PROTOCOL)
